# 🎯 Credit Risk API — QA Automation Master Suite

**Role:** Elite QA Automation Engineer × Credit Risk Domain Expert
**Engine:** XGBoost (156 trees) + WoE Transform → FICO Score (300–850)
**API:** `POST /api/v1/predict` | `POST /api/v1/enrich` | `GET /api/v1/enrich/stats`

## FICO Decision Thresholds (`predict.py#L82-102`)
| FICO Range | Risk Tier | Decision |
|---|---|---|
| ≥ 740 | LOW | APPROVED |
| 670 – 739 | MEDIUM_LOW | APPROVED_CONDITIONAL |
| 580 – 669 | MEDIUM_HIGH | APPROVED_CONDITIONAL |
| 570 – 579 | HIGH | MANUAL_REVIEW *(Gray Zone)* |
| < 570 | HIGH | REJECTED |

## Test Classes
1. **Setup & Infrastructure** — API health, config validation, invariant audit
2. **Happy Path Baselines** — Prime, Standard, Sub-prime profiles
3. **Adversarial Contradictory Profiles** — Semantic adversarial inputs
4. **Boundary Value Analysis** — FICO thresholds 740 / 670 / 580 / 570
5. **Gray Zone (Buffer Zone)** — 570–579 MANUAL_REVIEW vs REJECTED
6. **Data Pipeline Robustness** — Outliers, nulls, unknown categories
7. **ML Monotonicity & Pricing Arithmetic** — Constraint violations, PMT formula
8. **Enrich API CRUD** — Save, read, label, stats, export
9. **Statistical Report** — Full pass/fail/error breakdown with failure analysis

---
> ⚠️ **STRICT QA RULE**: Assertions are NEVER softened to make failing tests pass.
> A failed test = a successful discovery. Document discrepancies, do not hide them.

---
## Cell 1 — Environment Bootstrap & Test Runner Infrastructure

In [1]:
"""
Bootstrap: imports, configuration, shared harness.
"""
import sys, os, math, copy, time, traceback, json, datetime, warnings
from typing import Any, Callable
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

try:
    import requests
    from dotenv import load_dotenv
    load_dotenv(os.path.join(PROJECT_ROOT, ".env"), override=False)
except ImportError as e:
    raise SystemError(f"Missing dependency: {e}. Run: pip install requests python-dotenv")

BASE_URL = (os.getenv("TEST_API_BASE_URL") or os.getenv("VITE_API_BASE_URL", "http://127.0.0.1:8000")).rstrip("/")
PREDICT_URL   = f"{BASE_URL}/api/v1/predict"
ENRICH_URL    = f"{BASE_URL}/api/v1/enrich"
HEALTH_URL    = f"{BASE_URL}/health"
RECORDS_URL   = f"{BASE_URL}/api/v1/enrich/records"
STATS_URL     = f"{BASE_URL}/api/v1/enrich/stats"
EXPORT_URL    = f"{BASE_URL}/api/v1/enrich/export"
REQUEST_TIMEOUT = 20

print(f"API Base URL : {BASE_URL}")
print(f"Predict URL  : {PREDICT_URL}")
print(f"Enrich URL   : {ENRICH_URL}")

APR_MIN = 6.0
APR_MAX = 35.0   # config.yaml max_rate=35.0 (NOTE: conftest.py incorrectly uses 24.0)
FICO_MIN = 300
FICO_MAX = 850
TIER_APPROVED         = 740
TIER_CONDITIONAL_HIGH = 670
TIER_CONDITIONAL_LOW  = 580
TIER_GRAY_ZONE_LOW    = 570

# GROUND TRUTH from predict.py (NOT conftest.py — conftest has a bug for HIGH tier)
VALID_TIER_DECISIONS = {
    "LOW":         {"APPROVED"},
    "MEDIUM_LOW":  {"APPROVED_CONDITIONAL"},
    "MEDIUM_HIGH": {"APPROVED_CONDITIONAL"},
    "HIGH":        {"MANUAL_REVIEW", "REJECTED"},  # dual state per predict.py#L98-102
}

_BASELINE: dict[str, Any] = {
    "person_age": 28, "person_income": 65_000, "person_home_ownership": "RENT",
    "person_emp_length": 4.0, "loan_intent": "PERSONAL", "loan_grade": "B",
    "loan_amnt": 10_000, "loan_int_rate": 11.14, "loan_percent_income": 0.15,
    "cb_person_default_on_file": "N", "cb_person_cred_hist_length": 3,
    "gender": "MALE", "marital_status": "SINGLE", "education_level": "BACHELOR",
    "employment_type": "FULL_TIME", "loan_to_income_ratio": 0.15,
    "debt_to_income_ratio": 0.25, "credit_utilization_ratio": 0.35, "past_delinquencies": 0,
}

def payload(**overrides) -> dict:
    p = copy.deepcopy(_BASELINE)
    p.update(overrides)
    return p

RESULTS: list[dict] = []
RUN_START = datetime.datetime.now()

def record(test_id, name, category, passed, error="", actual=None, expected=None, elapsed_ms=0.0):
    status = "PASS" if passed else "FAIL"
    RESULTS.append({"id": test_id, "name": name, "category": category, "status": status,
                    "error": error, "actual": actual, "expected": expected, "elapsed_ms": round(elapsed_ms, 1)})
    icon = "✅" if passed else "❌"
    print(f"  {icon} [{test_id}] {name} ({elapsed_ms:.0f}ms)")
    if not passed:
        print(f"       FAILURE: {error}")
        if expected is not None:
            print(f"       Expected: {expected}  |  Actual: {actual}")

def run_test(test_id, name, category, fn):
    t0 = time.perf_counter()
    try:
        fn()
        elapsed = (time.perf_counter() - t0) * 1000
        record(test_id, name, category, passed=True, elapsed_ms=elapsed)
    except AssertionError as e:
        elapsed = (time.perf_counter() - t0) * 1000
        record(test_id, name, category, passed=False, error=str(e), elapsed_ms=elapsed)
    except Exception as e:
        elapsed = (time.perf_counter() - t0) * 1000
        RESULTS.append({"id": test_id, "name": name, "category": category, "status": "ERROR",
                        "error": f"{type(e).__name__}: {e}", "actual": None, "expected": None,
                        "elapsed_ms": round(elapsed, 1)})
        print(f"  💥 [{test_id}] {name} -> ERROR: {type(e).__name__}: {e}")

def assert_universal(response):
    """Enforce 6 baseline invariants on every /predict 200 response."""
    assert response.status_code == 200, f"Expected HTTP 200, got {response.status_code}. Body: {response.text[:300]}"
    body = response.json()
    assert body.get("success") is True, "'success' must be True"
    ass = body["credit_risk_assessment"]
    pd_score = ass["pd_score"]
    assert 0.0 <= pd_score <= 1.0, f"pd_score {pd_score} outside [0,1]"
    cs = ass["credit_score"]
    assert FICO_MIN <= cs <= FICO_MAX, f"credit_score {cs} outside [{FICO_MIN},{FICO_MAX}]"
    tier = ass["risk_tier"]
    dec = ass["decision"]
    valid_decisions = VALID_TIER_DECISIONS.get(tier, set())
    assert dec in valid_decisions, f"Mixed state: tier={tier!r} -> valid={valid_decisions}, got {dec!r}"
    pricing = ass.get("pricing_recommendation")
    if pricing:
        apr = pricing["recommended_interest_rate"]
        assert APR_MIN <= apr <= APR_MAX, f"APR {apr}% violates statutory bounds [{APR_MIN}, {APR_MAX}]"
        if dec == "REJECTED":
            assert pricing["max_credit_limit"] == 0.0, f"REJECTED must have limit=0.0, got {pricing['max_credit_limit']}"
            assert pricing["limit_status"] == "REJECTED"
        if dec == "MANUAL_REVIEW":
            assert pricing["max_credit_limit"] <= 3_000.0, f"MANUAL_REVIEW limit must be <= $3000. Got {pricing['max_credit_limit']}"
    return ass

print("Bootstrap complete.")
print(f"Run start: {RUN_START.isoformat()}")


API Base URL : http://127.0.0.1:8000
Predict URL  : http://127.0.0.1:8000/api/v1/predict
Enrich URL   : http://127.0.0.1:8000/api/v1/enrich
Bootstrap complete.
Run start: 2026-08-23T14:30:40.097734


---
## Class 1 — Infrastructure & Invariant Audit

In [2]:
print("\n" + "="*60)
print("CLASS 1: Infrastructure & Invariant Audit")
print("="*60)
CAT = "C1_Infrastructure"

def tc_inf_01():
    r = requests.get(HEALTH_URL, timeout=5)
    assert r.status_code == 200, f"/health returned {r.status_code}"
    assert r.json().get("status") == "healthy"
run_test("TC-INF-01", "Health Check — /health returns 200 + healthy", CAT, tc_inf_01)

def tc_inf_02():
    r = requests.get(BASE_URL + "/", timeout=5)
    assert r.status_code == 200
    body = r.json()
    assert "service" in body and "docs" in body
run_test("TC-INF-02", "Root endpoint returns service metadata", CAT, tc_inf_02)

def tc_inf_03():
    """
    AUDIT: conftest.py _VALID_TIER_DECISION_PAIRS['HIGH'] = 'REJECTED' only.
    predict.py#L98-100 shows HIGH -> MANUAL_REVIEW for scores 570-579 (Gray Zone).
    This test validates predict.py as ground truth and exposes the conftest bug.
    """
    p = payload(loan_grade="E", person_income=35_000, loan_amnt=8_000,
                loan_to_income_ratio=0.23, loan_percent_income=0.23,
                debt_to_income_ratio=0.42, loan_int_rate=18.5,
                person_emp_length=1.5, person_home_ownership="RENT",
                cb_person_default_on_file="N", loan_intent="PERSONAL")
    r = requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT)
    assert r.status_code == 200
    ass = r.json()["credit_risk_assessment"]
    tier, dec, cs = ass["risk_tier"], ass["decision"], ass["credit_score"]
    valid = VALID_TIER_DECISIONS.get(tier, set())
    assert dec in valid, f"predict.py invariant violated: tier={tier!r}, decision={dec!r}, valid={valid}"
    print(f"     [AUDIT] credit_score={cs}, tier={tier}, decision={dec}")
    if tier == "HIGH" and dec == "MANUAL_REVIEW":
        assert 570 <= cs <= 579, f"Gray zone MANUAL_REVIEW must have score in [570,579], got {cs}"
        print(f"     [AUDIT] Confirmed Gray Zone: score={cs} in [570,579]")
run_test("TC-INF-03", "[AUDIT] conftest HIGH-tier invariant incomplete — predict.py exposes dual state", CAT, tc_inf_03)

def tc_inf_04():
    r = requests.get(f"{BASE_URL}/docs", timeout=5)
    assert r.status_code == 200
    assert "swagger" in r.text.lower() or "openapi" in r.text.lower()
run_test("TC-INF-04", "Swagger /docs endpoint is accessible", CAT, tc_inf_04)

def tc_inf_05():
    r = requests.get(f"{BASE_URL}/openapi.json", timeout=5)
    assert r.status_code == 200
    schema = r.json()
    assert "/api/v1/predict" in schema.get("paths", {}), "predict path missing from OpenAPI schema"
    assert "/api/v1/enrich" in schema.get("paths", {}), "enrich path missing from OpenAPI schema"
run_test("TC-INF-05", "OpenAPI schema includes /predict and /enrich paths", CAT, tc_inf_05)

print(f"Class 1 done. Total: {len([r for r in RESULTS if r['category']==CAT])} tests.")



CLASS 1: Infrastructure & Invariant Audit
  ✅ [TC-INF-01] Health Check — /health returns 200 + healthy (14ms)
  ✅ [TC-INF-02] Root endpoint returns service metadata (30ms)


     [AUDIT] credit_score=562, tier=HIGH, decision=REJECTED
  ✅ [TC-INF-03] [AUDIT] conftest HIGH-tier invariant incomplete — predict.py exposes dual state (172ms)
  ✅ [TC-INF-04] Swagger /docs endpoint is accessible (6ms)
  ✅ [TC-INF-05] OpenAPI schema includes /predict and /enrich paths (91ms)
Class 1 done. Total: 5 tests.


---
## Class 2 — Happy Path Baselines

In [3]:
print("\n" + "="*60)
print("CLASS 2: Happy Path Baselines")
print("="*60)
CAT = "C2_HappyPath"

def tc_hp_01():
    """Prime Applicant: Grade A, OWN, EDUCATION, low DTI -> APPROVED + waterfall check."""
    p = payload(person_income=95_000, loan_amnt=10_000, loan_grade="A",
                loan_int_rate=7.5, loan_to_income_ratio=0.105, loan_percent_income=0.105,
                debt_to_income_ratio=0.18, person_emp_length=8.0,
                person_home_ownership="OWN", cb_person_default_on_file="N", loan_intent="EDUCATION")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    assert ass["credit_score"] >= TIER_APPROVED, f"Prime must score >= {TIER_APPROVED}. Got: {ass['credit_score']}"
    assert ass["risk_tier"] == "LOW"
    assert ass["decision"] == "APPROVED"
    assert ass["pd_score"] < 0.20, f"Prime PD must be < 0.20. Got: {ass['pd_score']}"
    pr = ass["pricing_recommendation"]
    assert pr["base_rate"] == 6.5
    assert pr["risk_spread"] == 1.0, f"LOW spread must be 1.0, got {pr['risk_spread']}"
    assert pr["capital_discount"] == -0.5, f"OWN discount must be -0.5, got {pr['capital_discount']}"
    assert pr["intent_adjustment"] == -0.3, f"EDUCATION adj must be -0.3, got {pr['intent_adjustment']}"
    expected_apr = round(6.5 + 1.0 - 0.5 - 0.3, 2)
    assert abs(pr["recommended_interest_rate"] - expected_apr) < 0.01, f"APR must be {expected_apr}%, got {pr['recommended_interest_rate']}%"
    assert pr["max_credit_limit"] <= 50_000.0
    assert pr["limit_status"] == "WITHIN_LIMIT"
run_test("TC-HP-01", "Prime Applicant — Grade A, OWN, EDUCATION -> APPROVED + APR=6.7%", CAT, tc_hp_01)

def tc_hp_02():
    """Standard baseline (initialFormData) — must not crash."""
    ass = assert_universal(requests.post(PREDICT_URL, json=copy.deepcopy(_BASELINE), timeout=REQUEST_TIMEOUT))
    assert ass["decision"] in ("APPROVED_CONDITIONAL", "MANUAL_REVIEW", "REJECTED"), f"Standard B must not be APPROVED. Got: {ass['decision']}"
run_test("TC-HP-02", "Standard Applicant — initialFormData baseline (no crash + valid tier)", CAT, tc_hp_02)

def tc_hp_03():
    """Sub-prime: Grade G, Y-default, DTI=0.85 -> REJECTED + limit=0."""
    p = payload(loan_grade="G", person_income=25_000, loan_amnt=20_000,
                loan_to_income_ratio=0.80, loan_percent_income=0.80,
                debt_to_income_ratio=0.85, loan_int_rate=24.0,
                person_emp_length=0.5, person_home_ownership="RENT",
                cb_person_default_on_file="Y", loan_intent="DEBTCONSOLIDATION")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    assert ass["decision"] == "REJECTED", f"Worst-case must be REJECTED. Got: {ass['decision']}"
    assert ass["credit_score"] < 570, f"Rejected must score < 570. Got: {ass['credit_score']}"
    assert ass["pd_score"] > 0.50, f"Worst-case PD must > 0.50. Got: {ass['pd_score']}"
    pr = ass["pricing_recommendation"]
    assert pr["max_credit_limit"] == 0.0
    assert pr["limit_status"] == "REJECTED"
run_test("TC-HP-03", "Sub-prime — Grade G, Y-default, DTI=0.85 -> REJECTED + limit=0", CAT, tc_hp_03)

def tc_hp_04():
    """HOMEIMPROVEMENT intent +0.2%, MORTGAGE discount -0.25%."""
    p = payload(loan_grade="B", person_income=75_000, loan_amnt=12_000,
                loan_to_income_ratio=0.16, loan_percent_income=0.16,
                debt_to_income_ratio=0.22, loan_int_rate=10.5,
                person_emp_length=6.0, person_home_ownership="MORTGAGE",
                cb_person_default_on_file="N", loan_intent="HOMEIMPROVEMENT")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    pr = ass["pricing_recommendation"]
    assert pr["intent_adjustment"] == 0.2, f"HOMEIMPROVEMENT adj must be +0.2%, got {pr['intent_adjustment']}"
    assert pr["capital_discount"] == -0.25, f"MORTGAGE discount must be -0.25%, got {pr['capital_discount']}"
run_test("TC-HP-04", "HOMEIMPROVEMENT +0.2% intent adj + MORTGAGE -0.25% capital discount", CAT, tc_hp_04)

def tc_hp_05():
    ass = assert_universal(requests.post(PREDICT_URL, json=payload(loan_intent="MEDICAL"), timeout=REQUEST_TIMEOUT))
    adj = ass["pricing_recommendation"]["intent_adjustment"]
    assert adj == 0.3, f"MEDICAL intent_adjustment must be +0.3%, got {adj}"
run_test("TC-HP-05", "MEDICAL intent -> +0.3% intent_adjustment applied correctly", CAT, tc_hp_05)

def tc_hp_06():
    ass = assert_universal(requests.post(PREDICT_URL, json=payload(loan_intent="VENTURE"), timeout=REQUEST_TIMEOUT))
    adj = ass["pricing_recommendation"]["intent_adjustment"]
    assert adj == 0.5, f"VENTURE intent_adjustment must be +0.5%, got {adj}"
run_test("TC-HP-06", "VENTURE intent -> +0.5% intent_adjustment applied correctly", CAT, tc_hp_06)

print(f"Class 2 done. Total: {len([r for r in RESULTS if r['category']==CAT])} tests.")



CLASS 2: Happy Path Baselines


  ✅ [TC-HP-01] Prime Applicant — Grade A, OWN, EDUCATION -> APPROVED + APR=6.7% (142ms)


  ✅ [TC-HP-02] Standard Applicant — initialFormData baseline (no crash + valid tier) (120ms)


  ✅ [TC-HP-03] Sub-prime — Grade G, Y-default, DTI=0.85 -> REJECTED + limit=0 (121ms)


  ✅ [TC-HP-04] HOMEIMPROVEMENT +0.2% intent adj + MORTGAGE -0.25% capital discount (118ms)


  ✅ [TC-HP-05] MEDICAL intent -> +0.3% intent_adjustment applied correctly (119ms)


  ✅ [TC-HP-06] VENTURE intent -> +0.5% intent_adjustment applied correctly (122ms)
Class 2 done. Total: 6 tests.


---
## Class 3 — Adversarial Contradictory Profiles

In [ ]:
print("\n" + "="*60)
print("CLASS 3: Adversarial Contradictory Profiles")
print("="*60)
CAT = "C3_Adversarial"

def tc_adv_01():
    """Grade A + Prior Default: WoE for Y must override Grade A. Auto-APPROVED = critical failure."""
    p = payload(loan_grade="A", cb_person_default_on_file="Y",
                loan_int_rate=7.0, person_income=100_000, loan_amnt=15_000,
                loan_to_income_ratio=0.15, loan_percent_income=0.15,
                debt_to_income_ratio=0.20, person_emp_length=10.0,
                person_home_ownership="OWN", loan_intent="PERSONAL")                                                                                                                                                                                                                                                                         
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))         
    assert ass["decision"] != "APPROVED", (
        f"CRITICAL: Grade-A + prior default MUST NOT be auto-approved. Got: {ass['decision']}")
    assert ass["pd_score"] > 0.15, f"Prior default must elevate PD > 0.15. Got: {ass['pd_score']}"
    neg_feats = [f["feature"] for f in ass.get("top_factors", {}).get("negative_factors", [])]
    assert "cb_person_default_on_file" in neg_feats, f"cb_person_default must be negative factor. Got: {neg_feats}"
run_test("TC-ADV-01", "Grade A + Prior Default — must NOT be auto-approved", CAT, tc_adv_01)

def tc_adv_02():
    """High Income + DTI=0.92: Capacity (5C) must dominate over absolute income."""
    p = payload(person_income=200_000, loan_amnt=50_000, loan_grade="C",
                loan_int_rate=14.0, loan_to_income_ratio=0.25, loan_percent_income=0.25,
                debt_to_income_ratio=0.92, person_emp_length=5.0,
                person_home_ownership="RENT", cb_person_default_on_file="N",
                loan_intent="DEBTCONSOLIDATION")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    assert ass["decision"] in ("MANUAL_REVIEW", "REJECTED"), f"DTI=0.92 must not approve. Got: {ass['decision']}"
    assert ass["pd_score"] > 0.30, f"Catastrophic DTI must have PD > 0.30. Got: {ass['pd_score']}"
    pr = ass["pricing_recommendation"]
    assert abs(pr["intent_adjustment"] - 0.8) < 0.001, f"DEBTCONSOLIDATION adj must be +0.8%, got {pr['intent_adjustment']}"
    neg_feats = [f["feature"] for f in ass.get("top_factors", {}).get("negative_factors", [])]
    assert "debt_to_income_ratio" in neg_feats, f"DTI must be negative factor. Got: {neg_feats}"
run_test("TC-ADV-02", "High Income + Catastrophic DTI=0.92 -> must not approve", CAT, tc_adv_02)

def tc_adv_03():
    """Grade G overrides clean default record, high income, low LTI."""
    p = payload(loan_grade="G", cb_person_default_on_file="N",
                person_income=150_000, loan_amnt=8_000,
                loan_to_income_ratio=0.053, loan_percent_income=0.053,
                debt_to_income_ratio=0.15, loan_int_rate=22.0,
                person_emp_length=12.0, person_home_ownership="OWN", loan_intent="PERSONAL")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    assert ass["decision"] in ("MANUAL_REVIEW", "REJECTED"), f"Grade G must not approve. Got: {ass['decision']}"
    assert ass["pd_score"] > 0.35, f"Grade G PD must > 0.35. Got: {ass['pd_score']}"
    neg_feats = [f["feature"] for f in ass.get("top_factors", {}).get("negative_factors", [])]
    assert "loan_grade" in neg_feats, f"loan_grade must be primary negative factor. Got: {neg_feats}"
run_test("TC-ADV-03", "Grade G + positive signals — loan_grade WoE must dominate", CAT, tc_adv_03)

def tc_adv_04():
    """VENTURE + emp_length=0 + LTI=0.50 masked by Grade A — must not auto-approve."""
    p = payload(loan_grade="A", loan_intent="VENTURE", employment_type="UNEMPLOYED",
                person_emp_length=0.0, person_income=30_000, loan_amnt=15_000,
                loan_to_income_ratio=0.50, loan_percent_income=0.50,
                debt_to_income_ratio=0.60, loan_int_rate=8.0,
                person_home_ownership="RENT", cb_person_default_on_file="N")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    assert ass["decision"] != "APPROVED", "Unemployed VENTURE LTI=0.50 must not auto-approve."
    assert ass["pd_score"] > 0.25
    assert abs(ass["pricing_recommendation"]["intent_adjustment"] - 0.5) < 0.001
run_test("TC-ADV-04", "VENTURE + emp_length=0 + LTI=0.50 masked by Grade A", CAT, tc_adv_04)

def tc_adv_05():
    """Monotonicity: PD(Grade G) > PD(Grade A) with all else equal."""
    base = dict(person_income=80_000, loan_amnt=10_000, loan_to_income_ratio=0.125,
                loan_percent_income=0.125, debt_to_income_ratio=0.25, loan_int_rate=12.0,
                person_emp_length=5.0, person_home_ownership="RENT",
                cb_person_default_on_file="N", loan_intent="PERSONAL")
    pd_a = requests.post(PREDICT_URL, json=payload(loan_grade="A", **base), timeout=REQUEST_TIMEOUT).json()["credit_risk_assessment"]["pd_score"]
    pd_g = requests.post(PREDICT_URL, json=payload(loan_grade="G", **base), timeout=REQUEST_TIMEOUT).json()["credit_risk_assessment"]["pd_score"]
    print(f"     [MONO] PD(Grade A)={pd_a:.4f} | PD(Grade G)={pd_g:.4f}")
    assert pd_g > pd_a, f"MONOTONICITY VIOLATION: PD(G)={pd_g:.4f} must > PD(A)={pd_a:.4f}"
run_test("TC-ADV-05", "Monotonicity — PD(Grade G) > PD(Grade A) with all else equal", CAT, tc_adv_05)

def tc_adv_06():
    """Monotonicity: PD(DTI=0.85) > PD(DTI=0.15)."""
    base = dict(loan_grade="C", person_income=60_000, loan_amnt=10_000,
                loan_to_income_ratio=0.167, loan_percent_income=0.167,
                loan_int_rate=14.0, person_emp_length=4.0,
                person_home_ownership="RENT", cb_person_default_on_file="N", loan_intent="PERSONAL")
    pd_l = requests.post(PREDICT_URL, json=payload(debt_to_income_ratio=0.15, **base), timeout=REQUEST_TIMEOUT).json()["credit_risk_assessment"]["pd_score"]
    pd_h = requests.post(PREDICT_URL, json=payload(debt_to_income_ratio=0.85, **base), timeout=REQUEST_TIMEOUT).json()["credit_risk_assessment"]["pd_score"]
    print(f"     [MONO] PD(DTI=0.15)={pd_l:.4f} | PD(DTI=0.85)={pd_h:.4f}")
    assert pd_h > pd_l, f"MONOTONICITY VIOLATION: PD(DTI=0.85)={pd_h:.4f} must > PD(DTI=0.15)={pd_l:.4f}"
run_test("TC-ADV-06", "Monotonicity — PD(DTI=0.85) > PD(DTI=0.15) with all else equal", CAT, tc_adv_06)

print(f"Class 3 done. Total: {len([r for r in RESULTS if r['category']==CAT])} tests.")



CLASS 3: Adversarial Contradictory Profiles


  ✅ [TC-ADV-01] Grade A + Prior Default — must NOT be auto-approved (120ms)


  ✅ [TC-ADV-02] High Income + Catastrophic DTI=0.92 -> must not approve (115ms)


  ❌ [TC-ADV-03] Grade G + positive signals — loan_grade WoE must dominate (129ms)
       FAILURE: Grade G must not approve. Got: APPROVED


  ✅ [TC-ADV-04] VENTURE + emp_length=0 + LTI=0.50 masked by Grade A (116ms)


     [MONO] PD(Grade A)=0.1041 | PD(Grade G)=0.0965
  ❌ [TC-ADV-05] Monotonicity — PD(Grade G) > PD(Grade A) with all else equal (232ms)
       FAILURE: MONOTONICITY VIOLATION: PD(G)=0.0965 must > PD(A)=0.1041


     [MONO] PD(DTI=0.15)=0.0736 | PD(DTI=0.85)=0.4554
  ✅ [TC-ADV-06] Monotonicity — PD(DTI=0.85) > PD(DTI=0.15) with all else equal (235ms)
Class 3 done. Total: 6 tests.


---
## Class 4 — Boundary Value Analysis (FICO Thresholds)

In [5]:
print("\n" + "="*60)
print("CLASS 4: Boundary Value Analysis — FICO Thresholds")
print("="*60)
CAT = "C4_BVA"

def tc_bva_01():
    """Near 740 boundary. Decision must be APPROVED or APPROVED_CONDITIONAL, coherent with score."""
    p = payload(loan_grade="A", person_income=80_000, loan_amnt=8_000,
                loan_to_income_ratio=0.10, loan_percent_income=0.10,
                debt_to_income_ratio=0.22, loan_int_rate=8.5,
                person_emp_length=6.0, person_home_ownership="MORTGAGE",
                cb_person_default_on_file="N", loan_intent="PERSONAL")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    cs, tier, dec = ass["credit_score"], ass["risk_tier"], ass["decision"]
    print(f"     [BVA-740] credit_score={cs}, tier={tier}, decision={dec}")
    assert cs >= TIER_CONDITIONAL_HIGH, f"Near-prime must score >= {TIER_CONDITIONAL_HIGH}. Got: {cs}"
    assert dec in ("APPROVED", "APPROVED_CONDITIONAL"), f"Near-prime must not be MANUAL_REVIEW/REJECTED. Got: {dec}"
    if cs >= TIER_APPROVED:
        assert tier == "LOW" and dec == "APPROVED", f"score={cs}>=740 must be LOW/APPROVED. Got tier={tier} dec={dec}"
run_test("TC-BVA-01", "FICO Boundary 740 — APPROVED vs APPROVED_CONDITIONAL coherence", CAT, tc_bva_01)

def tc_bva_02():
    """Near 670 boundary. Whatever score lands, tier/decision must be consistent."""
    p = payload(loan_grade="B", person_income=70_000, loan_amnt=12_000,
                loan_to_income_ratio=0.17, loan_percent_income=0.17,
                debt_to_income_ratio=0.30, loan_int_rate=11.0,
                person_emp_length=5.0, person_home_ownership="RENT",
                cb_person_default_on_file="N", loan_intent="PERSONAL")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    cs, tier, dec = ass["credit_score"], ass["risk_tier"], ass["decision"]
    print(f"     [BVA-670] credit_score={cs}, tier={tier}, decision={dec}")
    if cs >= TIER_CONDITIONAL_HIGH:
        assert tier in ("LOW", "MEDIUM_LOW") and dec in ("APPROVED", "APPROVED_CONDITIONAL")
    elif cs >= TIER_CONDITIONAL_LOW:
        assert tier == "MEDIUM_HIGH" and dec == "APPROVED_CONDITIONAL"
run_test("TC-BVA-02", "FICO Boundary 670 — MEDIUM_LOW vs MEDIUM_HIGH coherence", CAT, tc_bva_02)

def tc_bva_03():
    """Near 580 boundary. APPROVED_CONDITIONAL vs MANUAL_REVIEW/REJECTED."""
    p = payload(loan_grade="D", person_income=45_000, loan_amnt=12_000,
                loan_to_income_ratio=0.267, loan_percent_income=0.267,
                debt_to_income_ratio=0.45, loan_int_rate=16.5,
                person_emp_length=2.0, person_home_ownership="RENT",
                cb_person_default_on_file="N", loan_intent="PERSONAL")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    cs, dec = ass["credit_score"], ass["decision"]
    print(f"     [BVA-580] credit_score={cs}, decision={dec}")
    assert dec in ("APPROVED_CONDITIONAL", "MANUAL_REVIEW", "REJECTED")
    if dec == "REJECTED":
        assert ass["pricing_recommendation"]["max_credit_limit"] == 0.0
run_test("TC-BVA-03", "FICO Boundary 580 — APPROVED_CONDITIONAL vs MANUAL_REVIEW/REJECTED", CAT, tc_bva_03)

def tc_bva_04():
    """LTI exactly at LOW tier cap: max_credit_limit = min(100k*0.40, 50k) = 40k. WITHIN_LIMIT."""
    p = payload(loan_grade="A", person_income=100_000, loan_amnt=40_000,
                loan_to_income_ratio=0.40, loan_percent_income=0.40,
                debt_to_income_ratio=0.20, loan_int_rate=8.0,
                person_emp_length=7.0, person_home_ownership="OWN",
                cb_person_default_on_file="N", loan_intent="PERSONAL")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    assert ass["decision"] == "APPROVED"
    pr = ass["pricing_recommendation"]
    assert abs(pr["max_credit_limit"] - 40_000.0) < 0.01, f"Limit at LTI boundary must be $40,000. Got: {pr['max_credit_limit']}"
    assert pr["limit_status"] == "WITHIN_LIMIT"
run_test("TC-BVA-04", "LTI exactly at LOW tier cap (40%) -> WITHIN_LIMIT + limit=$40,000", CAT, tc_bva_04)

def tc_bva_05():
    """LTI $1 over LOW cap: limit_status must flip to EXCEEDS_RECOMMENDED_LIMIT, ML decision unchanged."""
    p = payload(loan_grade="A", person_income=100_000, loan_amnt=40_001,
                loan_to_income_ratio=0.40001, loan_percent_income=0.40001,
                debt_to_income_ratio=0.20, loan_int_rate=8.0,
                person_emp_length=7.0, person_home_ownership="OWN",
                cb_person_default_on_file="N", loan_intent="PERSONAL")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    assert ass["decision"] == "APPROVED", "Limit breach must NOT downgrade ML decision."
    pr = ass["pricing_recommendation"]
    assert pr["limit_status"] == "EXCEEDS_RECOMMENDED_LIMIT", f"$1 over cap -> EXCEEDS_RECOMMENDED_LIMIT. Got: {pr['limit_status']}"
run_test("TC-BVA-05", "LTI $1 over LOW cap -> EXCEEDS_RECOMMENDED_LIMIT, ML unchanged", CAT, tc_bva_05)

def tc_bva_06():
    """FICO floor clamp — worst-case must not produce credit_score < 300."""
    p = payload(loan_grade="G", person_income=10_000, loan_amnt=9_000,
                loan_to_income_ratio=0.90, loan_percent_income=0.90,
                debt_to_income_ratio=0.95, loan_int_rate=25.0,
                person_emp_length=0.0, person_home_ownership="RENT",
                cb_person_default_on_file="Y", loan_intent="DEBTCONSOLIDATION")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    assert ass["credit_score"] >= FICO_MIN, f"FICO must be clamped to minimum {FICO_MIN}. Got: {ass['credit_score']}"
run_test("TC-BVA-06", "FICO floor clamp — worst-case must not go below 300", CAT, tc_bva_06)

print(f"Class 4 done. Total: {len([r for r in RESULTS if r['category']==CAT])} tests.")



CLASS 4: Boundary Value Analysis — FICO Thresholds
     [BVA-740] credit_score=752, tier=LOW, decision=APPROVED
  ✅ [TC-BVA-01] FICO Boundary 740 — APPROVED vs APPROVED_CONDITIONAL coherence (118ms)


     [BVA-670] credit_score=588, tier=MEDIUM_HIGH, decision=APPROVED_CONDITIONAL
  ✅ [TC-BVA-02] FICO Boundary 670 — MEDIUM_LOW vs MEDIUM_HIGH coherence (178ms)
     [BVA-580] credit_score=508, decision=REJECTED
  ✅ [TC-BVA-03] FICO Boundary 580 — APPROVED_CONDITIONAL vs MANUAL_REVIEW/REJECTED (151ms)


  ❌ [TC-BVA-04] LTI exactly at LOW tier cap (40%) -> WITHIN_LIMIT + limit=$40,000 (128ms)
       FAILURE: 
  ❌ [TC-BVA-05] LTI $1 over LOW cap -> EXCEEDS_RECOMMENDED_LIMIT, ML unchanged (114ms)
       FAILURE: Limit breach must NOT downgrade ML decision.


  ✅ [TC-BVA-06] FICO floor clamp — worst-case must not go below 300 (126ms)
Class 4 done. Total: 6 tests.


---
## Class 5 — Gray Zone / Buffer Zone (570–579)

In [6]:
print("\n" + "="*60)
print("CLASS 5: Gray Zone Buffer Zone — FICO 570-579")
print("="*60)
CAT = "C5_GrayZone"

def tc_gz_01():
    """
    predict.py#L98-100: scores 570-579 -> HIGH tier -> MANUAL_REVIEW.
    The conftest.py bug claims HIGH -> REJECTED always.
    This test validates the actual implementation.
    """
    p = payload(loan_grade="D", person_income=55_000, loan_amnt=10_000,
                loan_to_income_ratio=0.18, loan_percent_income=0.18,
                debt_to_income_ratio=0.35, loan_int_rate=15.0,
                person_emp_length=3.0, person_home_ownership="RENT",
                cb_person_default_on_file="N", loan_intent="PERSONAL")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    cs, tier, dec = ass["credit_score"], ass["risk_tier"], ass["decision"]
    pr = ass["pricing_recommendation"]
    print(f"     [GRAY ZONE PROBE] credit_score={cs}, tier={tier}, decision={dec}")
    if TIER_GRAY_ZONE_LOW <= cs <= (TIER_CONDITIONAL_LOW - 1):  # 570-579
        assert tier == "HIGH", f"Gray zone score {cs} must be HIGH tier, got {tier}"
        assert dec == "MANUAL_REVIEW", (
            f"CRITICAL: Gray zone score {cs} (570-579) must be MANUAL_REVIEW, got {dec}. "
            "predict.py#L98-100 explicitly routes 570-579 to MANUAL_REVIEW.")
        assert pr["max_credit_limit"] <= 3_000.0, f"MANUAL_REVIEW gray zone limit <= $3,000. Got: {pr['max_credit_limit']}"
        assert pr["limit_status"] == "MANUAL_REVIEW"
        print(f"     [GRAY ZONE] Confirmed MANUAL_REVIEW for score {cs} (570-579)")
    elif cs < TIER_GRAY_ZONE_LOW:
        assert dec == "REJECTED", f"score={cs} < 570 must be REJECTED. Got: {dec}"
        print(f"     [GRAY ZONE] Profile below gray zone at {cs}. REJECTED confirmed.")
    else:
        print(f"     [GRAY ZONE] Profile above gray zone at {cs} ({dec}). Consider tuning payload.")
run_test("TC-GZ-01", "Gray Zone probe — FICO 570-579 must route to MANUAL_REVIEW (not REJECTED)", CAT, tc_gz_01)

def tc_gz_02():
    """MANUAL_REVIEW credit limit must be capped at $3,000 (gray zone)."""
    p = payload(loan_grade="D", person_income=55_000, loan_amnt=10_000,
                loan_to_income_ratio=0.18, loan_percent_income=0.18,
                debt_to_income_ratio=0.35, loan_int_rate=15.0,
                person_emp_length=3.0, person_home_ownership="RENT",
                cb_person_default_on_file="N", loan_intent="PERSONAL")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    dec = ass["decision"]
    pr = ass["pricing_recommendation"]
    print(f"     [GZ-CAP] cs={ass['credit_score']}, decision={dec}, limit={pr['max_credit_limit']}")
    if dec == "MANUAL_REVIEW":
        assert pr["max_credit_limit"] <= 3_000.0, f"Gray zone limit must be <= $3,000. Got: {pr['max_credit_limit']}"
run_test("TC-GZ-02", "Gray zone MANUAL_REVIEW credit limit capped at $3,000", CAT, tc_gz_02)

def tc_gz_03():
    """MANUAL_REVIEW -> limit_status='MANUAL_REVIEW'. REJECTED -> limit_status='REJECTED' + limit=0."""
    p = payload(loan_grade="E", person_income=40_000, loan_amnt=7_000,
                loan_to_income_ratio=0.175, loan_percent_income=0.175,
                debt_to_income_ratio=0.38, loan_int_rate=17.0,
                person_emp_length=2.0, person_home_ownership="RENT",
                cb_person_default_on_file="N", loan_intent="PERSONAL")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    dec = ass["decision"]
    pr = ass["pricing_recommendation"]
    print(f"     [GZ-DIST] cs={ass['credit_score']}, decision={dec}, limit_status={pr['limit_status']}, limit={pr['max_credit_limit']}")
    if dec == "REJECTED":
        assert pr["limit_status"] == "REJECTED"
        assert pr["max_credit_limit"] == 0.0
    elif dec == "MANUAL_REVIEW":
        assert pr["limit_status"] == "MANUAL_REVIEW"
        assert 0 < pr["max_credit_limit"] <= 3_000.0
run_test("TC-GZ-03", "Gray zone pricing distinction — MANUAL_REVIEW vs REJECTED limit_status", CAT, tc_gz_03)

print(f"Class 5 done. Total: {len([r for r in RESULTS if r['category']==CAT])} tests.")



CLASS 5: Gray Zone Buffer Zone — FICO 570-579
     [GRAY ZONE PROBE] credit_score=628, tier=MEDIUM_HIGH, decision=APPROVED_CONDITIONAL
     [GRAY ZONE] Profile above gray zone at 628 (APPROVED_CONDITIONAL). Consider tuning payload.
  ✅ [TC-GZ-01] Gray Zone probe — FICO 570-579 must route to MANUAL_REVIEW (not REJECTED) (130ms)


     [GZ-CAP] cs=628, decision=APPROVED_CONDITIONAL, limit=9900.0
  ✅ [TC-GZ-02] Gray zone MANUAL_REVIEW credit limit capped at $3,000 (119ms)
     [GZ-DIST] cs=582, decision=APPROVED_CONDITIONAL, limit_status=WITHIN_LIMIT, limit=7200.0
  ✅ [TC-GZ-03] Gray zone pricing distinction — MANUAL_REVIEW vs REJECTED limit_status (125ms)
Class 5 done. Total: 3 tests.


---
## Class 6 — Data Pipeline Robustness

In [7]:
print("\n" + "="*60)
print("CLASS 6: Data Pipeline Robustness")
print("="*60)
CAT = "C6_Robustness"

def tc_rob_01():
    p = copy.deepcopy(_BASELINE); p.pop("loan_grade")
    r = requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT)
    assert r.status_code == 422, f"Missing loan_grade must return 422. Got: {r.status_code}"
    locs = [" -> ".join(str(l) for l in e.get("loc", [])) for e in r.json().get("detail", [])]
    assert any("loan_grade" in l for l in locs), f"422 must mention loan_grade. Got: {locs}"
run_test("TC-ROB-01", "Missing required field (loan_grade) -> HTTP 422 with field name", CAT, tc_rob_01)

def tc_rob_02():
    required = {"person_age", "person_income", "person_home_ownership", "person_emp_length",
                "loan_intent", "loan_grade", "loan_amnt", "loan_int_rate", "loan_percent_income",
                "cb_person_default_on_file", "cb_person_cred_hist_length"}
    r = requests.post(PREDICT_URL, json={}, timeout=REQUEST_TIMEOUT)
    assert r.status_code == 422
    missing_fields = {e["loc"][-1] for e in r.json().get("detail", []) if e.get("type") in ("missing", "value_error.missing")}
    assert required.issubset(missing_fields), f"422 must enumerate all 11 required fields. Missing: {required - missing_fields}"
run_test("TC-ROB-02", "Empty body -> 422 with all 11 required fields enumerated", CAT, tc_rob_02)

def tc_rob_03():
    p = copy.deepcopy(_BASELINE); p["loan_amnt"] = "ten thousand"
    r = requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT)
    assert r.status_code == 422, f"String loan_amnt must return 422. Got: {r.status_code}"
run_test("TC-ROB-03", "Wrong data type (loan_amnt='ten thousand') -> HTTP 422", CAT, tc_rob_03)

def tc_rob_04():
    """loan_int_rate=-5.0: no ge=0 constraint in schema — must not 500."""
    r = requests.post(PREDICT_URL, json=payload(loan_int_rate=-5.0), timeout=REQUEST_TIMEOUT)
    assert r.status_code != 500, f"CRITICAL P1 BUG: HTTP 500 on loan_int_rate=-5.0. Response: {r.text[:300]}"
    assert r.status_code in (200, 422), f"Expected 200 or 422, got {r.status_code}"
run_test("TC-ROB-04", "Negative interest rate (-5.0) -> must not cause HTTP 500", CAT, tc_rob_04)

def tc_rob_05():
    """Unknown loan_grade='Z': WoE binner must use woe_map.get(value, 0.0) fallback."""
    r = requests.post(PREDICT_URL, json=payload(loan_grade="Z"), timeout=REQUEST_TIMEOUT)
    assert r.status_code != 500, "HTTP 500 on unknown loan_grade='Z' indicates no WoE fallback."
    if r.status_code == 200:
        ass = assert_universal(r); assert isinstance(ass["pd_score"], float)
run_test("TC-ROB-05", "Unknown loan_grade='Z' -> graceful WoE fallback (not HTTP 500)", CAT, tc_rob_05)

def tc_rob_06():
    """$10M income: outside quantile bins. Hard cap $50K must prevent $4M credit line."""
    p = payload(person_income=10_000_000, loan_amnt=50_000, loan_grade="A",
                loan_to_income_ratio=0.005, loan_percent_income=0.005,
                debt_to_income_ratio=0.01, loan_int_rate=7.0,
                person_emp_length=15.0, person_home_ownership="OWN",
                cb_person_default_on_file="N", loan_intent="PERSONAL")
    r = requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT)
    assert r.status_code != 500, "HTTP 500 on $10M income indicates WoE binner cannot handle quantile outliers."
    if r.status_code == 200:
        ass = assert_universal(r)
        assert ass["pricing_recommendation"]["max_credit_limit"] <= 50_000.0, (
            f"Hard cap $50,000 must prevent $4M credit line. Got: {ass['pricing_recommendation']['max_credit_limit']}")
run_test("TC-ROB-06", "Extreme outlier income ($10M) -> WoE handles + $50K hard cap", CAT, tc_rob_06)

def tc_rob_07():
    """LTI=6.25 (loan=$500K on $80K income): WoE must clamp to last known bin."""
    p = payload(person_income=80_000, loan_amnt=500_000, loan_grade="B",
                loan_to_income_ratio=6.25, loan_percent_income=6.25,
                debt_to_income_ratio=0.30, loan_int_rate=13.0,
                person_emp_length=5.0, person_home_ownership="RENT",
                cb_person_default_on_file="N", loan_intent="PERSONAL")
    r = requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT)
    assert r.status_code != 500, "HTTP 500 on extreme LTI=6.25 indicates WoE cannot handle out-of-distribution value."
    if r.status_code == 200:
        ass = assert_universal(r)
        assert ass["pd_score"] > 0.40, f"Extreme LTI=6.25 must produce high PD. Got: {ass['pd_score']}"
run_test("TC-ROB-07", "Extreme LTI=6.25 (loan=$500K) -> WoE clamps, no HTTP 500", CAT, tc_rob_07)

def tc_rob_08():
    """income=$1: near-zero income must not cause division-by-zero. REJECTED expected."""
    p = payload(person_income=1, loan_amnt=500, loan_grade="D",
                loan_to_income_ratio=500.0, loan_percent_income=500.0,
                debt_to_income_ratio=0.5, loan_int_rate=17.0,
                person_emp_length=1.0, person_home_ownership="RENT",
                cb_person_default_on_file="N", loan_intent="PERSONAL")
    r = requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT)
    assert r.status_code != 500, "HTTP 500 on income=$1 indicates division-by-zero or numeric overflow."
    if r.status_code == 200:
        ass = assert_universal(r); assert ass["decision"] == "REJECTED"
run_test("TC-ROB-08", "Near-zero income ($1) -> no division-by-zero + REJECTED", CAT, tc_rob_08)

def tc_rob_09():
    p = copy.deepcopy(_BASELINE); p["debt_to_income_ratio"] = None
    r = requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT)
    assert r.status_code != 500, "HTTP 500 on null optional field debt_to_income_ratio."
    assert r.status_code in (200, 422)
run_test("TC-ROB-09", "Explicit null for optional field (debt_to_income_ratio) -> 200 or 422", CAT, tc_rob_09)

def tc_rob_10():
    """person_emp_length=0.0 must not be treated as NaN by the preprocessor."""
    p = payload(person_emp_length=0.0, loan_grade="B", person_income=55_000, loan_amnt=8_000,
                loan_to_income_ratio=0.145, loan_percent_income=0.145,
                debt_to_income_ratio=0.28, loan_int_rate=12.0,
                person_home_ownership="RENT", cb_person_default_on_file="N", loan_intent="PERSONAL")
    r = requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT)
    assert r.status_code != 500, "HTTP 500 on person_emp_length=0.0 indicates WoE binner treats 0.0 as NaN."
    if r.status_code == 200:
        ass = assert_universal(r); assert isinstance(ass["pd_score"], float)
run_test("TC-ROB-10", "Zero employment length (0.0) -> not treated as NaN, no HTTP 500", CAT, tc_rob_10)

def tc_rob_11():
    """Path traversal in 'task' query param. Must not leak Python tracebacks."""
    r = requests.post(PREDICT_URL, json=_BASELINE, params={"task": "../../../etc/passwd"}, timeout=REQUEST_TIMEOUT)
    assert r.status_code in (400, 404, 422, 500)
    detail = str(r.json().get("detail", ""))
    assert "Traceback" not in detail, f"Response must not disclose tracebacks. Got: {detail[:300]}"
run_test("TC-ROB-11", "Path traversal in task param -> structured error, no traceback leak", CAT, tc_rob_11)

def tc_rob_12():
    """Statutory APR cap (<=35%) — worst case Grade G + DEBTCONSOLIDATION."""
    p = payload(loan_grade="G", person_income=30_000, loan_amnt=5_000,
                loan_to_income_ratio=0.167, loan_percent_income=0.167,
                debt_to_income_ratio=0.70, loan_int_rate=25.0,
                person_emp_length=0.5, person_home_ownership="RENT",
                cb_person_default_on_file="Y", loan_intent="DEBTCONSOLIDATION")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    pr = ass["pricing_recommendation"]
    assert pr["recommended_interest_rate"] <= APR_MAX, (
        f"CRITICAL COMPLIANCE: APR={pr['recommended_interest_rate']}% exceeds statutory cap of {APR_MAX}%.")
    assert pr["risk_spread"] == 9.0, f"HIGH tier risk_spread must be 9.0. Got: {pr['risk_spread']}"
run_test("TC-ROB-12", "Statutory APR cap (<=35%) — worst case Grade G + DEBTCONSOLIDATION", CAT, tc_rob_12)

print(f"Class 6 done. Total: {len([r for r in RESULTS if r['category']==CAT])} tests.")



CLASS 6: Data Pipeline Robustness
  ✅ [TC-ROB-01] Missing required field (loan_grade) -> HTTP 422 with field name (7ms)
  ✅ [TC-ROB-02] Empty body -> 422 with all 11 required fields enumerated (7ms)
  ✅ [TC-ROB-03] Wrong data type (loan_amnt='ten thousand') -> HTTP 422 (6ms)


  ✅ [TC-ROB-04] Negative interest rate (-5.0) -> must not cause HTTP 500 (140ms)


  ✅ [TC-ROB-05] Unknown loan_grade='Z' -> graceful WoE fallback (not HTTP 500) (128ms)


  ✅ [TC-ROB-06] Extreme outlier income ($10M) -> WoE handles + $50K hard cap (123ms)


  ✅ [TC-ROB-07] Extreme LTI=6.25 (loan=$500K) -> WoE clamps, no HTTP 500 (125ms)


  ✅ [TC-ROB-08] Near-zero income ($1) -> no division-by-zero + REJECTED (120ms)


  ✅ [TC-ROB-09] Explicit null for optional field (debt_to_income_ratio) -> 200 or 422 (115ms)


  ✅ [TC-ROB-10] Zero employment length (0.0) -> not treated as NaN, no HTTP 500 (123ms)
  ✅ [TC-ROB-11] Path traversal in task param -> structured error, no traceback leak (8ms)


  ✅ [TC-ROB-12] Statutory APR cap (<=35%) — worst case Grade G + DEBTCONSOLIDATION (124ms)
Class 6 done. Total: 12 tests.


---
## Class 7 — ML Monotonicity, Pricing Arithmetic & PMT Simulation

In [8]:
print("\n" + "="*60)
print("CLASS 7: ML Monotonicity, Pricing Arithmetic & PMT Simulation")
print("="*60)
CAT = "C7_PricingML"

# Intent adjustment tests
INTENT_ADJUSTMENTS = {
    "EDUCATION": -0.3, "PERSONAL": 0.0, "HOMEIMPROVEMENT": 0.2,
    "MEDICAL": 0.3, "VENTURE": 0.5, "DEBTCONSOLIDATION": 0.8
}
for intent_name, exp_adj in INTENT_ADJUSTMENTS.items():
    def make_intent_test(iname, eadj):
        def t():
            ass = assert_universal(requests.post(PREDICT_URL, json=payload(loan_intent=iname), timeout=REQUEST_TIMEOUT))
            actual = ass["pricing_recommendation"]["intent_adjustment"]
            assert abs(actual - eadj) < 0.001, f"{iname} intent_adjustment must be {eadj}%, got {actual}%"
        return t
    run_test(f"TC-PRC-01-{intent_name}", f"Intent adjustment: {intent_name} -> {exp_adj:+.1f}%", CAT, make_intent_test(intent_name, exp_adj))

# Home ownership discount tests
HOME_DISCOUNTS = {"OWN": -0.5, "MORTGAGE": -0.25, "RENT": 0.0, "OTHER": 0.0}
for home_type, exp_disc in HOME_DISCOUNTS.items():
    def make_home_test(htype, edisc):
        def t():
            ass = assert_universal(requests.post(PREDICT_URL, json=payload(person_home_ownership=htype), timeout=REQUEST_TIMEOUT))
            actual = ass["pricing_recommendation"]["capital_discount"]
            assert abs(actual - edisc) < 0.001, f"{htype} capital_discount must be {edisc}%, got {actual}%"
        return t
    run_test(f"TC-PRC-02-{home_type}", f"Capital discount: {home_type} -> {exp_disc:+.2f}%", CAT, make_home_test(home_type, exp_disc))

def tc_prc_03():
    """PMT = P * [r(1+r)^n / ((1+r)^n - 1)]. total_interest = PMT*n - P >= 0."""
    loan_principal = 10_000.0
    p = payload(person_income=95_000, loan_amnt=loan_principal, loan_grade="A",
                loan_int_rate=7.5, loan_to_income_ratio=0.105, loan_percent_income=0.105,
                debt_to_income_ratio=0.18, person_emp_length=8.0,
                person_home_ownership="OWN", cb_person_default_on_file="N", loan_intent="EDUCATION")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    pr = ass["pricing_recommendation"]
    apr, n, pmt, ti = pr["recommended_interest_rate"], pr["loan_term_months"], pr["monthly_payment_estimate"], pr["total_interest_estimate"]
    assert n == 36, f"Default loan term must be 36 months. Got: {n}"
    mr = apr / (12 * 100)
    expected_pmt = loan_principal * (mr * (1 + mr) ** n / ((1 + mr) ** n - 1))
    assert abs(pmt - expected_pmt) / expected_pmt < 0.01, f"PMT mismatch. Expected {expected_pmt:.2f}, got {pmt:.2f}. APR={apr}%, n={n}"
    assert ti >= 0, f"total_interest_estimate must be >= 0. Got: {ti}"
    expected_ti = round(pmt * n - loan_principal, 2)
    assert abs(ti - expected_ti) < 1.0, f"total_interest mismatch. Expected ~{expected_ti:.2f}, got {ti:.2f}"
    print(f"     [PMT] APR={apr}% | PMT={pmt:.2f} | Expected={expected_pmt:.2f} | TotalInterest={ti:.2f}")
run_test("TC-PRC-03", "PMT formula arithmetic verification — PMT = P*[r(1+r)^n/((1+r)^n-1)]", CAT, tc_prc_03)

def tc_prc_04():
    """For prime applicant: loan_grade='A' and cb_default='N' WoE contributions must be positive."""
    p = payload(person_income=95_000, loan_amnt=10_000, loan_grade="A",
                loan_int_rate=7.5, loan_to_income_ratio=0.105, loan_percent_income=0.105,
                debt_to_income_ratio=0.18, person_emp_length=8.0,
                person_home_ownership="OWN", cb_person_default_on_file="N", loan_intent="EDUCATION")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    contributions = ass.get("contributions", {})
    assert len(ass.get("top_factors", {}).get("positive_factors", [])) >= 1, "Prime must have >= 1 positive factor."
    if "loan_grade" in contributions:
        assert contributions["loan_grade"] > 0, f"loan_grade='A' must have positive WoE. Got: {contributions['loan_grade']}"
    if "cb_person_default_on_file" in contributions:
        assert contributions["cb_person_default_on_file"] > 0, f"cb_default='N' must have positive WoE. Got: {contributions['cb_person_default_on_file']}"
run_test("TC-PRC-04", "WoE contribution signs — Grade A / default=N must be positive", CAT, tc_prc_04)

def tc_prc_05():
    """All output fields form a logically consistent vector for worst-case profile."""
    p = payload(loan_grade="G", person_income=25_000, loan_amnt=20_000,
                loan_to_income_ratio=0.80, loan_percent_income=0.80,
                debt_to_income_ratio=0.85, loan_int_rate=24.0,
                person_emp_length=0.5, person_home_ownership="RENT",
                cb_person_default_on_file="Y", loan_intent="DEBTCONSOLIDATION")
    ass = assert_universal(requests.post(PREDICT_URL, json=p, timeout=REQUEST_TIMEOUT))
    assert ass["pd_score"] > 0.50, f"Worst-case PD must > 0.50. Got: {ass['pd_score']}"
    assert ass["credit_score"] < TIER_GRAY_ZONE_LOW, f"Worst-case score must < {TIER_GRAY_ZONE_LOW}. Got: {ass['credit_score']}"
    assert ass["risk_tier"] == "HIGH" and ass["decision"] == "REJECTED"
    pr = ass["pricing_recommendation"]
    assert pr["max_credit_limit"] == 0.0 and pr["limit_status"] == "REJECTED"
    assert pr["recommended_interest_rate"] <= APR_MAX
run_test("TC-PRC-05", "Worst-case coherence vector — all fields consistent", CAT, tc_prc_05)

def tc_prc_06():
    """ECOA-protected / dropped features must NOT appear in contributions."""
    dropped_features = {"gender", "marital_status", "education_level", "employment_type",
                        "credit_utilization_ratio", "past_delinquencies", "cb_person_cred_hist_length",
                        "person_age", "loan_percent_income"}
    ass = assert_universal(requests.post(PREDICT_URL, json=_BASELINE, timeout=REQUEST_TIMEOUT))
    contributions = ass.get("contributions", {})
    actual_keys = set(contributions.keys())
    ecoa_in_contributions = dropped_features & actual_keys
    assert not ecoa_in_contributions, f"ECOA/dropped features must not be in contributions. Found: {ecoa_in_contributions}"
    print(f"     [CONTRIBUTIONS] Keys: {sorted(actual_keys)}")
run_test("TC-PRC-06", "Contribution keys — ECOA-protected/dropped features absent from response", CAT, tc_prc_06)

print(f"Class 7 done. Total: {len([r for r in RESULTS if r['category']==CAT])} tests.")



CLASS 7: ML Monotonicity, Pricing Arithmetic & PMT Simulation


  ✅ [TC-PRC-01-EDUCATION] Intent adjustment: EDUCATION -> -0.3% (125ms)


  ✅ [TC-PRC-01-PERSONAL] Intent adjustment: PERSONAL -> +0.0% (134ms)


  ✅ [TC-PRC-01-HOMEIMPROVEMENT] Intent adjustment: HOMEIMPROVEMENT -> +0.2% (122ms)


  ✅ [TC-PRC-01-MEDICAL] Intent adjustment: MEDICAL -> +0.3% (132ms)


...

  ✅ [TC-PRC-02-OTHER] Capital discount: OTHER -> +0.00% (124ms)


     [PMT] APR=6.7% | PMT=307.40 | Expected=307.40 | TotalInterest=1066.40
  ✅ [TC-PRC-03] PMT formula arithmetic verification — PMT = P*[r(1+r)^n/((1+r)^n-1)] (123ms)


  ✅ [TC-PRC-04] WoE contribution signs — Grade A / default=N must be positive (130ms)


  ✅ [TC-PRC-05] Worst-case coherence vector — all fields consistent (125ms)


     [CONTRIBUTIONS] Keys: ['cb_person_default_on_file', 'debt_to_income_ratio', 'loan_amnt', 'loan_grade', 'loan_int_rate', 'loan_intent', 'loan_term_months', 'loan_to_income_ratio', 'person_emp_length', 'person_home_ownership', 'person_income']
  ✅ [TC-PRC-06] Contribution keys — ECOA-protected/dropped features absent from response (124ms)
Class 7 done. Total: 14 tests.


---
## Class 8 — Enrich API CRUD Validation

In [9]:
print("\n" + "="*60)
print("CLASS 8: Enrich API CRUD Validation")
print("="*60)
CAT = "C8_EnrichAPI"
_saved_client_id = [None]  # use list to allow mutation in nested functions

def tc_enr_01():
    enrich_payload = {"application": _BASELINE, "loan_status": -1,
                      "ml_pd_score": 0.23, "ml_credit_score": 685, "ml_decision": "APPROVED_CONDITIONAL"}
    r = requests.post(ENRICH_URL, json=enrich_payload, timeout=REQUEST_TIMEOUT)
    assert r.status_code == 200, f"POST /enrich must return 200. Got: {r.status_code} {r.text[:200]}"
    body = r.json()
    assert body.get("success") is True
    assert "client_ID" in body and "saved_to" in body and "created_at" in body
    _saved_client_id[0] = body["client_ID"]
    print(f"     [ENRICH] Saved client_ID: {_saved_client_id[0]}")
run_test("TC-ENR-01", "POST /enrich — save new record, returns client_ID + saved_to + created_at", CAT, tc_enr_01)

def tc_enr_02():
    r = requests.get(RECORDS_URL, params={"page": 1, "page_size": 5}, timeout=REQUEST_TIMEOUT)
    assert r.status_code == 200
    body = r.json()
    assert body.get("success") is True
    assert "total" in body and "records" in body
    assert isinstance(body["records"], list)
    assert body["page"] == 1 and body["page_size"] == 5
    print(f"     [ENRICH] total={body['total']}, returned={len(body['records'])}")
run_test("TC-ENR-02", "GET /enrich/records — paginated list with total + page metadata", CAT, tc_enr_02)

def tc_enr_03():
    r = requests.get(STATS_URL, timeout=REQUEST_TIMEOUT)
    assert r.status_code == 200
    body = r.json()
    assert body.get("success") is True
    assert "total_records" in body and "default_rate_pct" in body
    assert "retrain_threshold" in body and "ready_for_retrain" in body
    assert body["total_records"] >= 0
    assert 0 <= body["default_rate_pct"] <= 100
    print(f"     [STATS] total={body['total_records']}, default_rate={body['default_rate_pct']}%, retrain_ready={body['ready_for_retrain']}")
run_test("TC-ENR-03", "GET /enrich/stats — total + default_rate_pct + retrain fields", CAT, tc_enr_03)

def tc_enr_04():
    if not _saved_client_id[0]:
        raise AssertionError("No client_ID from TC-ENR-01; cannot test labelling.")
    r = requests.patch(f"{ENRICH_URL}/records/{_saved_client_id[0]}/label",
                       params={"loan_status": 1}, timeout=REQUEST_TIMEOUT)
    assert r.status_code == 200, f"PATCH /label must return 200. Got: {r.status_code} {r.text[:200]}"
    body = r.json()
    assert body.get("success") is True and body.get("loan_status") == 1
    assert body.get("client_ID") == _saved_client_id[0]
    print(f"     [LABEL] Labelled {_saved_client_id[0]} as loan_status=1")
run_test("TC-ENR-04", "PATCH /enrich/records/{id}/label — update loan_status to 1 (default)", CAT, tc_enr_04)

def tc_enr_05():
    r = requests.patch(f"{ENRICH_URL}/records/FAKE_ID/label", params={"loan_status": 99}, timeout=REQUEST_TIMEOUT)
    assert r.status_code == 400, f"Invalid loan_status=99 must return 400. Got: {r.status_code}"
run_test("TC-ENR-05", "PATCH /label with invalid loan_status=99 -> HTTP 400", CAT, tc_enr_05)

def tc_enr_06():
    r = requests.patch(f"{ENRICH_URL}/records/DOES_NOT_EXIST/label", params={"loan_status": 0}, timeout=REQUEST_TIMEOUT)
    assert r.status_code == 404, f"Non-existent client_ID must return 404. Got: {r.status_code}"
run_test("TC-ENR-06", "PATCH /label for non-existent client_ID -> HTTP 404", CAT, tc_enr_06)

def tc_enr_07():
    enrich_payload = {"application": _BASELINE, "loan_status": -99,
                      "ml_pd_score": 0.50, "ml_credit_score": 600, "ml_decision": "MANUAL_REVIEW"}
    r = requests.post(ENRICH_URL, json=enrich_payload, timeout=REQUEST_TIMEOUT)
    assert r.status_code == 422, f"loan_status=-99 must return 422. Got: {r.status_code}"
run_test("TC-ENR-07", "POST /enrich with loan_status=-99 -> HTTP 422 (out of range)", CAT, tc_enr_07)

def tc_enr_08():
    r = requests.get(EXPORT_URL, timeout=REQUEST_TIMEOUT)
    if r.status_code == 404:
        print("     [EXPORT] No data yet — 404 is acceptable on empty dataset."); return
    assert r.status_code == 200
    assert "text/csv" in r.headers.get("content-type", "")
    assert "attachment" in r.headers.get("Content-Disposition", "").lower()
    lines = r.text.strip().split("\n")
    assert len(lines) >= 2, "CSV must have header + at least 1 data row."
    assert "client_ID" in lines[0] or "client_id" in lines[0].lower()
    print(f"     [EXPORT] CSV rows: {len(lines)-1}, header: {lines[0][:80]}...")
run_test("TC-ENR-08", "GET /enrich/export — returns valid CSV with correct headers", CAT, tc_enr_08)

print(f"Class 8 done. Total: {len([r for r in RESULTS if r['category']==CAT])} tests.")



CLASS 8: Enrich API CRUD Validation
     [ENRICH] Saved client_ID: ENRICH_5A09EB34D8D6606B3EC4E07217121A4A
  ✅ [TC-ENR-01] POST /enrich — save new record, returns client_ID + saved_to + created_at (13ms)


     [ENRICH] total=1, returned=1
  ✅ [TC-ENR-02] GET /enrich/records — paginated list with total + page metadata (35ms)
     [STATS] total=1, default_rate=0.0%, retrain_ready=False
  ✅ [TC-ENR-03] GET /enrich/stats — total + default_rate_pct + retrain fields (23ms)
     [LABEL] Labelled ENRICH_5A09EB34D8D6606B3EC4E07217121A4A as loan_status=1
  ✅ [TC-ENR-04] PATCH /enrich/records/{id}/label — update loan_status to 1 (default) (16ms)
  ✅ [TC-ENR-05] PATCH /label with invalid loan_status=99 -> HTTP 400 (14ms)
  ✅ [TC-ENR-06] PATCH /label for non-existent client_ID -> HTTP 404 (35ms)
  ✅ [TC-ENR-07] POST /enrich with loan_status=-99 -> HTTP 422 (out of range) (17ms)


     [EXPORT] CSV rows: 1, header: client_ID,person_age,person_income,person_home_ownership,person_emp_length,loan_...
  ✅ [TC-ENR-08] GET /enrich/export — returns valid CSV with correct headers (18ms)
Class 8 done. Total: 8 tests.


---
## Cell 9 — Statistical Report & Failure Analysis

In [10]:
from IPython.display import display, HTML
import json as _json

RUN_END = datetime.datetime.now()
DURATION = (RUN_END - RUN_START).total_seconds()

total   = len(RESULTS)
passed  = sum(1 for r in RESULTS if r["status"] == "PASS")
failed  = sum(1 for r in RESULTS if r["status"] == "FAIL")
errored = sum(1 for r in RESULTS if r["status"] == "ERROR")
pass_rate = (passed / total * 100) if total > 0 else 0
failures = [r for r in RESULTS if r["status"] in ("FAIL", "ERROR")]

categories = {}
for r in RESULTS:
    cat = r["category"]
    if cat not in categories:
        categories[cat] = {"pass": 0, "fail": 0, "error": 0}
    categories[cat][r["status"].lower()] += 1

status_color = "#2ed573" if pass_rate == 100 else ("#ffa502" if pass_rate >= 80 else "#ff4757")
status_label = "ALL PASS" if pass_rate == 100 else ("MOSTLY PASS" if pass_rate >= 80 else "NEEDS ATTENTION")

failure_rows = ""
for f in failures:
    bg = "#ff4757" if f["status"] == "FAIL" else "#ffa502"
    failure_rows += f"""
        <tr style="border-bottom: 1px solid #333;">
            <td style="padding:8px; color:{bg}; font-weight:bold;">{f["status"]}</td>
            <td style="padding:8px; color:#74b9ff; font-family:monospace; font-size:13px;">{f["id"]}</td>
            <td style="padding:8px; color:#dfe4ea;">{f["name"]}</td>
            <td style="padding:8px; color:#ff6b81; font-size:12px; max-width:400px; word-wrap:break-word;">{str(f["error"])[:300]}</td>
            <td style="padding:8px; color:#a4b0be;">{f["elapsed_ms"]:.0f}ms</td>
        </tr>"""

cat_display = {
    "C1_Infrastructure": "Class 1: Infrastructure & Invariant Audit",
    "C2_HappyPath": "Class 2: Happy Path Baselines",
    "C3_Adversarial": "Class 3: Adversarial Contradictory Profiles",
    "C4_BVA": "Class 4: Boundary Value Analysis (FICO)",
    "C5_GrayZone": "Class 5: Gray Zone Buffer (570-579)",
    "C6_Robustness": "Class 6: Data Pipeline Robustness",
    "C7_PricingML": "Class 7: Pricing Arithmetic & PMT",
    "C8_EnrichAPI": "Class 8: Enrich API CRUD",
}
cat_rows = ""
for cat, counts in categories.items():
    n = counts["pass"] + counts["fail"] + counts["error"]
    cpr = counts["pass"] / n * 100 if n > 0 else 0
    color = "#2ed573" if cpr == 100 else ("#ffa502" if cpr >= 80 else "#ff4757")
    cat_rows += f"""
        <tr style="border-bottom:1px solid #2f3542;">
            <td style="padding:10px;color:#dfe4ea;">{cat_display.get(cat, cat)}</td>
            <td style="padding:10px;text-align:center;">{n}</td>
            <td style="padding:10px;text-align:center;color:#2ed573;">{counts["pass"]}</td>
            <td style="padding:10px;text-align:center;color:#ff4757;">{counts["fail"]}</td>
            <td style="padding:10px;text-align:center;color:#ffa502;">{counts["error"]}</td>
            <td style="padding:10px;text-align:center;color:{color};font-weight:bold;">{cpr:.0f}%</td>
        </tr>"""

failure_section = f"""
  <h2 style="color:#ff4757;font-size:18px;margin-bottom:10px;">Failure Analysis ({len(failures)} issues)</h2>
  <table style="width:100%;border-collapse:collapse;margin-bottom:25px;font-size:13px;">
    <thead><tr style="background:#2f3542;color:#74b9ff;">
      <th style="padding:8px;text-align:left;">Status</th>
      <th style="padding:8px;text-align:left;">Test ID</th>
      <th style="padding:8px;text-align:left;">Test Name</th>
      <th style="padding:8px;text-align:left;">Failure Reason</th>
      <th style="padding:8px;text-align:left;">Time</th>
    </tr></thead>
    <tbody>{failure_rows}</tbody>
  </table>""" if failures else """
  <div style="color:#2ed573;font-size:16px;padding:15px;background:#2f3542;border-radius:8px;margin-bottom:25px;">
    All tests passed! No failures or errors detected.
  </div>"""

html_report = f"""
<div style="background:#1e1e1e;color:#fff;padding:30px;border-radius:12px;
            font-family:'Segoe UI',Tahoma,sans-serif;box-shadow:0 8px 24px rgba(0,0,0,0.5);
            border:1px solid #444;max-width:1200px;">
  <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:20px;border-bottom:2px solid #2f3542;padding-bottom:15px;">
    <div>
      <h1 style="color:#2ed573;margin:0;font-size:26px;">Credit Risk API — QA Statistical Report</h1>
      <p style="color:#a4b0be;margin:5px 0 0 0;font-size:14px;">
        Run completed: {RUN_END.strftime("%Y-%m-%d %H:%M:%S")} | Duration: {DURATION:.1f}s
      </p>
    </div>
    <div style="text-align:center;">
      <div style="font-size:48px;font-weight:bold;color:{status_color};">{pass_rate:.0f}%</div>
      <div style="color:{status_color};font-size:14px;font-weight:bold;">{status_label}</div>
    </div>
  </div>
  <div style="display:flex;gap:15px;margin-bottom:25px;">
    <div style="background:#2f3542;padding:20px 30px;border-radius:10px;text-align:center;flex:1;">
      <div style="font-size:36px;font-weight:bold;color:#74b9ff;">{total}</div>
      <div style="color:#a4b0be;font-size:13px;margin-top:5px;">TOTAL TESTS</div>
    </div>
    <div style="background:#2f3542;padding:20px 30px;border-radius:10px;text-align:center;flex:1;">
      <div style="font-size:36px;font-weight:bold;color:#2ed573;">{passed}</div>
      <div style="color:#a4b0be;font-size:13px;margin-top:5px;">PASSED</div>
    </div>
    <div style="background:#2f3542;padding:20px 30px;border-radius:10px;text-align:center;flex:1;">
      <div style="font-size:36px;font-weight:bold;color:#ff4757;">{failed}</div>
      <div style="color:#a4b0be;font-size:13px;margin-top:5px;">FAILED</div>
    </div>
    <div style="background:#2f3542;padding:20px 30px;border-radius:10px;text-align:center;flex:1;">
      <div style="font-size:36px;font-weight:bold;color:#ffa502;">{errored}</div>
      <div style="color:#a4b0be;font-size:13px;margin-top:5px;">ERRORS</div>
    </div>
  </div>
  <h2 style="color:#dfe4ea;font-size:18px;margin-bottom:10px;">Results by Test Class</h2>
  <table style="width:100%;border-collapse:collapse;margin-bottom:25px;font-size:14px;">
    <thead><tr style="background:#2f3542;color:#74b9ff;">
      <th style="padding:10px;text-align:left;">Test Class</th>
      <th style="padding:10px;text-align:center;">Total</th>
      <th style="padding:10px;text-align:center;">Pass</th>
      <th style="padding:10px;text-align:center;">Fail</th>
      <th style="padding:10px;text-align:center;">Error</th>
      <th style="padding:10px;text-align:center;">Pass Rate</th>
    </tr></thead>
    <tbody>{cat_rows}</tbody>
  </table>
  {failure_section}
  <h2 style="color:#ffa502;font-size:18px;margin-bottom:10px;">Critical Engineering Findings</h2>
  <div style="background:#2f3542;padding:15px;border-radius:8px;margin-bottom:15px;border-left:4px solid #ffa502;">
    <b style="color:#ffa502;">FINDING-001: conftest.py Invariant Table Incomplete (HIGH Tier)</b>
    <p style="color:#dfe4ea;margin:8px 0;font-size:13px;">
      <b>File:</b> tests/conftest.py#L118-123<br/>
      <b>Bug:</b> _VALID_TIER_DECISION_PAIRS maps "HIGH" to "REJECTED" only.<br/>
      <b>Reality:</b> predict.py#L98-100 routes scores 570-579 to HIGH tier -> MANUAL_REVIEW (Gray Zone).<br/>
      <b>Impact:</b> universal_assertions() will incorrectly FAIL on valid gray-zone MANUAL_REVIEW responses.<br/>
      <b>Fix:</b> Update conftest.py: "HIGH": {{"MANUAL_REVIEW", "REJECTED"}}
    </p>
  </div>
  <div style="background:#2f3542;padding:15px;border-radius:8px;margin-bottom:15px;border-left:4px solid #ff6b81;">
    <b style="color:#ff6b81;">FINDING-002: conftest.py APR Cap Mismatch (24.0 vs 35.0)</b>
    <p style="color:#dfe4ea;margin:8px 0;font-size:13px;">
      <b>File:</b> tests/conftest.py#L127<br/>
      <b>Bug:</b> _APR_MAX=24.0 but config.yaml#L175 sets max_rate: 35.0.<br/>
      <b>Impact:</b> False failures for valid HIGH-tier APR values between 24-35%.<br/>
      <b>Fix:</b> Set _APR_MAX = 35.0 in conftest.py.
    </p>
  </div>
  <div style="background:#2f3542;padding:15px;border-radius:8px;margin-bottom:15px;border-left:4px solid #a29bfe;">
    <b style="color:#a29bfe;">FINDING-003: loan_int_rate Missing ge=0 Schema Constraint</b>
    <p style="color:#dfe4ea;margin:8px 0;font-size:13px;">
      <b>File:</b> services/api_server/app/schemas/loan.py#L14<br/>
      <b>Gap:</b> loan_int_rate: float has no ge=0.0 validation constraint.<br/>
      <b>Risk:</b> Negative rates reach the WoE binner and may crash with ValueError.<br/>
      <b>Fix:</b> Add ge=0.0 to the Field(...) declaration for loan_int_rate.
    </p>
  </div>
  <div style="border-top:1px solid #2f3542;padding-top:15px;margin-top:10px;display:flex;justify-content:space-between;color:#636e72;font-size:12px;">
    <span>QA Suite: tests/test_credit_risk_qa_suite.ipynb</span>
    <span>Model: XGBoost (156 trees, max_depth=5, scale_pos_weight=3.58)</span>
    <span>Generated: {RUN_END.isoformat()}</span>
  </div>
</div>
"""
display(HTML(html_report))

# Text summary
print("\n" + "="*70)
print("  CREDIT RISK API — QA FINAL SUMMARY")
print("="*70)
print(f"  Total Tests  : {total}")
print(f"  Passed       : {passed}  ({pass_rate:.1f}%)")
print(f"  Failed       : {failed}")
print(f"  Errors       : {errored}")
print(f"  Duration     : {DURATION:.1f}s")
print("-"*70)
if failures:
    print(f"\n  FAILURE ANALYSIS ({len(failures)} issues):")
    for i, f in enumerate(failures, 1):
        print(f"\n  [{i}] {f['status']} | {f['id']} | {f['name']}")
        print(f"      Reason: {str(f['error'])[:200]}")
else:
    print("\n  All tests passed. No failures or errors detected.")
print("\n  CRITICAL FINDINGS (DO NOT fix by adjusting assertions):")
print("  FINDING-001: conftest.py HIGH tier maps only to REJECTED — misses MANUAL_REVIEW (570-579)")
print("               Fix: tests/conftest.py#L122: 'HIGH': {'MANUAL_REVIEW', 'REJECTED'}")
print("  FINDING-002: conftest.py _APR_MAX=24.0 mismatches config.yaml max_rate=35.0")
print("               Fix: tests/conftest.py#L127: _APR_MAX = 35.0")
print("  FINDING-003: loan_int_rate schema missing ge=0 constraint")
print("               Fix: services/api_server/app/schemas/loan.py#L14: ge=0.0")
print("="*70)



  CREDIT RISK API — QA FINAL SUMMARY
  Total Tests  : 60
  Passed       : 56  (93.3%)
  Failed       : 4
  Errors       : 0
  Duration     : 6.7s
----------------------------------------------------------------------

  FAILURE ANALYSIS (4 issues):

  [1] FAIL | TC-ADV-03 | Grade G + positive signals — loan_grade WoE must dominate
      Reason: Grade G must not approve. Got: APPROVED

  [2] FAIL | TC-ADV-05 | Monotonicity — PD(Grade G) > PD(Grade A) with all else equal
      Reason: MONOTONICITY VIOLATION: PD(G)=0.0965 must > PD(A)=0.1041

  [3] FAIL | TC-BVA-04 | LTI exactly at LOW tier cap (40%) -> WITHIN_LIMIT + limit=$40,000
      Reason: 

  [4] FAIL | TC-BVA-05 | LTI $1 over LOW cap -> EXCEEDS_RECOMMENDED_LIMIT, ML unchanged
      Reason: Limit breach must NOT downgrade ML decision.

  CRITICAL FINDINGS (DO NOT fix by adjusting assertions):
  FINDING-001: conftest.py HIGH tier maps only to REJECTED — misses MANUAL_REVIEW (570-579)
               Fix: tests/conftest.py#L122: 'HIG

---
## Cell 10 — Export Results to JSON

In [11]:
import json as _json

report_path = os.path.join(os.getcwd(), "qa_report.json")
report_data = {
    "meta": {
        "suite": "test_credit_risk_qa_suite.ipynb",
        "api_url": BASE_URL,
        "run_start": RUN_START.isoformat(),
        "run_end": RUN_END.isoformat(),
        "duration_seconds": round(DURATION, 2),
        "model": "XGBoost credit_risk (156 trees, max_depth=5, scale_pos_weight=3.58)",
    },
    "summary": {"total": total, "passed": passed, "failed": failed, "errored": errored, "pass_rate_pct": round(pass_rate, 2)},
    "findings": [
        {"id": "FINDING-001", "severity": "HIGH", "file": "tests/conftest.py#L118-123",
         "title": "_VALID_TIER_DECISION_PAIRS[HIGH] is incomplete",
         "description": "conftest.py maps HIGH -> REJECTED only. predict.py#L98-100 routes scores 570-579 to HIGH -> MANUAL_REVIEW.",
         "fix": "Update conftest.py line 122 to: 'HIGH': {'MANUAL_REVIEW', 'REJECTED'}"},
        {"id": "FINDING-002", "severity": "MEDIUM", "file": "tests/conftest.py#L127",
         "title": "APR cap mismatch: conftest uses 24.0 but config.yaml sets 35.0",
         "description": "_APR_MAX=24.0 in conftest does not match config.yaml max_rate=35.0.",
         "fix": "Update conftest.py: _APR_MAX = 35.0"},
        {"id": "FINDING-003", "severity": "MEDIUM", "file": "services/api_server/app/schemas/loan.py#L14",
         "title": "loan_int_rate missing ge=0 constraint",
         "description": "loan_int_rate: float has no ge=0.0 validation. Negative rates bypass schema and may crash WoE binner.",
         "fix": "Add ge=0.0 to Field(...) declaration for loan_int_rate"},
    ],
    "results": [{"id": r["id"], "name": r["name"], "category": r["category"],
                 "status": r["status"], "error": r["error"], "elapsed_ms": r["elapsed_ms"]}
                for r in RESULTS],
}
with open(report_path, "w", encoding="utf-8") as f:
    _json.dump(report_data, f, indent=2, ensure_ascii=False, default=str)
print(f"Full results exported to: {report_path}")
print(f"Total test records: {len(RESULTS)}")
print(f"Findings documented: {len(report_data['findings'])}")


Full results exported to: C:\Users\OMEN\Desktop\Learn\FPT\DATN\tests\qa_report.json
Total test records: 60
Findings documented: 3
